# CREMA-D: extra visual views for the input ablation (Kaggle, CPU only)

Adds two views next to the original frames (A0: 2 fps, longest side 448), for experiment A:
- **A1**: 4 fps (twice the frames), same resolution;
- **A2**: face crops at A0's timestamps, 336 x 336 (OpenCV face detector, one box per clip).
A3 (full frames + face crops) reuses A0 and A2. The script is `analysis/preprocess_cremad_views.py` (with
`preprocess_cremad.py`, whose download and frame functions it uses), written into the notebook below.

**Running it**
1. *Accelerator*: **None**; *Internet*: **on**; *Add input*: the output of `prep_cremad.ipynb` (`cremad_mm/`).
2. **Save & Run All (Commit)**. About 1-2 hours (the videos are downloaded again). Clips that fail are marked
   `error: ...`; commit again with this notebook's output attached to retry only those.
3. Check `face_samples.jpg` (48 random crops; a red border = no face found, central crop) and the printed share of
   frames with a detected face. The output folder `/kaggle/working/cremad_views/` (about 1.5 GB) is an input of
   `probe_visual.ipynb` for A1-A3.

In [ ]:
%pip install -q "opencv-python-headless==4.10.0.84"
import shutil, subprocess
if not (shutil.which("ffmpeg") and shutil.which("ffprobe")):
    subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg", shell=True, check=True)

In [ ]:
%%writefile preprocess_cremad.py
"""Download CREMA-D, extract frames and 16 kHz audio, and write a manifest with the raters' votes per condition.

CREMA-D (Cao et al., 2014; https://github.com/CheyneyComputerScience/CREMA-D, ODbL 1.0): 7,442 clips of 91 actors
reading 12 fixed sentences, each rated by about 10 people in each of three conditions: audio only, video only,
audio-visual (different people per condition). Six labels: Anger, Disgust, Fear, Happy, Neutral, Sad.

Media come from the repository's Git LFS store through media.githubusercontent.com (no clone needed); the
per-rater responses (finishedResponses.csv) and actor table from raw.githubusercontent.com.

Output (paths relative to <out>, same layout as preprocess_chsims.py so the run notebooks can read it):
    <out>/frames/<actor>/<clip>/NNN.jpg     frames from the presentation video (VideoFlash/*.flv)
    <out>/audio/<actor>/<clip>.wav          mono 16 kHz from AudioWAV/*.wav
    <out>/manifest.csv                      one row per clip: actor, sentence, intended emotion and level,
                                            split (by actor), votes per condition, paths, status

Votes: `{cond}_{label}` counts and `{cond}_n` raters for cond in a (audio only), v (video only), av (audio-visual);
labels in LABELS order. Split: actors shuffled with a fixed seed, 70/10/20 % of actors to train/valid/test,
stratified by sex. Clips the dataset notes as broken are kept with status `known_bad`.

Usage (needs ffmpeg/ffprobe on PATH):
    python analysis/preprocess_cremad.py --out-dir /data/cremad_mm
    python analysis/preprocess_cremad.py --out-dir /tmp/cremad_mm --limit 20   # quick check
"""

from __future__ import annotations

import argparse
import io
import json
import shutil
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

REPO = "CheyneyComputerScience/CREMA-D"
MEDIA = f"https://media.githubusercontent.com/media/{REPO}/master"
RAW = f"https://raw.githubusercontent.com/{REPO}/master"
LABELS = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
CODE = {"A": "Anger", "D": "Disgust", "F": "Fear", "H": "Happy", "N": "Neutral", "S": "Sad"}
CONDITIONS = {1: "a", 2: "v", 3: "av"}  # queryType in finishedResponses.csv
SENTENCES = {"IEO": "It's eleven o'clock.", "TIE": "That is exactly what happened.", "IOM": "I'm on my way to the meeting.",
             "IWW": "I wonder what this is about.", "TAI": "The airplane is almost full.",
             "MTI": "Maybe tomorrow it will be cold.", "IWL": "I would like a new alarm clock.",
             "ITH": "I think I have a doctor's appointment.", "DFA": "Don't forget a jacket.",
             "ITS": "I think I've seen this before.", "TSI": "The surface is slick.",
             "WSI": "We'll stop in a couple of minutes."}
INTENDED = {"ANG": "Anger", "DIS": "Disgust", "FEA": "Fear", "HAP": "Happy", "NEU": "Neutral", "SAD": "Sad"}
LEVEL = {"LO": "low", "MD": "medium", "HI": "high", "XX": "unspecified"}
# probe and extract_frames as in preprocess_chsims.py (copied so this file runs on its own, e.g. on Kaggle)


def probe(path: Path) -> tuple[float, bool]:
    """Return (duration in seconds, whether the file has an audio stream)."""
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type", "-of", "json", str(path)],
        capture_output=True, text=True, check=True,
    )
    info = json.loads(out.stdout)
    duration = float(info.get("format", {}).get("duration", 0.0))
    has_audio = any(s.get("codec_type") == "audio" for s in info.get("streams", []))
    return duration, has_audio


def extract_frames(src: Path, dst: Path, fps: float, max_frames: int, max_side: int, duration: float) -> tuple[int, float]:
    """Sample frames at `fps`, lowered so a clip never yields more than `max_frames`."""
    if duration > 0:
        fps = min(fps, max_frames / duration)
    tmp = dst.with_name(dst.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    scale = f"scale={max_side}:{max_side}:force_original_aspect_ratio=decrease"
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(src), "-vf", f"fps={fps:.4f},{scale}", "-q:v", "3", str(tmp / "%03d.jpg")]
    subprocess.run(cmd, check=True)
    frames = sorted(tmp.glob("*.jpg"))
    if not frames:  # clip shorter than one sampling interval: keep its first frame
        subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(src), "-vf", scale, "-frames:v", "1",
                        "-q:v", "3", str(tmp / "001.jpg")], check=True)
        frames = sorted(tmp.glob("*.jpg"))
    for extra in frames[max_frames:]:
        extra.unlink()
    shutil.rmtree(dst, ignore_errors=True)
    tmp.rename(dst)
    return min(len(frames), max_frames), fps


KNOWN_BAD = {"1076_MTI_NEU_XX", "1076_MTI_SAD_XX", "1064_TIE_SAD_XX", "1064_IEO_DIS_MD"}  # per the dataset README


def fetch(url: str, tries: int = 5) -> bytes:
    for k in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(2 ** k)


def votes_table(responses: pd.DataFrame) -> pd.DataFrame:
    r = responses.drop_duplicates(["localid", "clipName", "queryType"])
    r = r[r.respEmo.isin(CODE)]
    counts = r.assign(label=r.respEmo.map(CODE)).groupby(["clipName", "queryType", "label"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=LABELS, fill_value=0)
    parts = []
    for q, cond in CONDITIONS.items():
        c = counts.xs(q, level="queryType")
        c.columns = [f"{cond}_{l}" for l in LABELS]
        c[f"{cond}_n"] = c.sum(axis=1)
        parts.append(c)
    return pd.concat(parts, axis=1).fillna(0).astype(int)


def split_actors(actors: pd.DataFrame, seed: int = 0) -> dict[int, str]:
    rng = np.random.default_rng(seed)
    out = {}
    for _, group in actors.groupby("Sex"):
        ids = rng.permutation(group.ActorID.to_numpy())
        n_train, n_valid = round(0.7 * len(ids)), round(0.1 * len(ids))
        out.update({int(a): "train" for a in ids[:n_train]})
        out.update({int(a): "valid" for a in ids[n_train:n_train + n_valid]})
        out.update({int(a): "test" for a in ids[n_train + n_valid:]})
    return out


def clip_table() -> pd.DataFrame:
    """One row per clip: actor, sentence, intended emotion and level, split by actor, votes per condition."""
    responses = pd.read_csv(io.BytesIO(fetch(f"{RAW}/finishedResponses.csv")), low_memory=False)
    actors = pd.read_csv(io.BytesIO(fetch(f"{RAW}/VideoDemographics.csv")))
    names = pd.read_csv(io.BytesIO(fetch(f"{RAW}/SentenceFilenames.csv")))["Filename"]

    clips = pd.DataFrame({"uid": names})
    parts = clips.uid.str.split("_", expand=True)
    clips["actor"] = parts[0].astype(int)
    clips["sentence_code"], clips["text"] = parts[1], parts[1].map(SENTENCES)
    clips["intended"], clips["level"] = parts[2].map(INTENDED), parts[3].map(LEVEL)
    clips["video_id"] = clips.actor.astype(str)  # grouping key for video-level bootstraps (one actor = one "video")
    clips["mode"] = clips.actor.map(split_actors(actors))
    clips = clips.merge(actors.rename(columns={"ActorID": "actor"}), on="actor", how="left")
    return clips.merge(votes_table(responses), left_on="uid", right_index=True, how="left")


def process_clip(row: dict, out_dir: Path, args: argparse.Namespace) -> dict:
    clip, actor = row["uid"], str(row["actor"])
    frames_rel, audio_rel = Path("frames") / actor / clip, Path("audio") / actor / f"{clip}.wav"
    result = {"uid": clip, "frames_dir": str(frames_rel), "audio_path": str(audio_rel), "n_frames": 0,
              "sample_fps": 0.0, "duration": 0.0, "status": "known_bad" if clip in KNOWN_BAD else "ok"}
    try:
        audio = out_dir / audio_rel
        if not audio.exists():
            audio.parent.mkdir(parents=True, exist_ok=True)
            tmp = audio.with_name(audio.stem + ".dl.wav")
            tmp.write_bytes(fetch(f"{MEDIA}/AudioWAV/{clip}.wav"))
            subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(tmp), "-ac", "1", "-ar", "16000",
                            "-c:a", "pcm_s16le", str(audio)], check=True, capture_output=True, text=True)
            tmp.unlink()
        frames_dir = out_dir / frames_rel
        existing = sorted(frames_dir.glob("*.jpg")) if frames_dir.exists() else []
        video = out_dir / "tmp_video" / f"{clip}.flv"
        if not existing or not (out_dir / frames_rel / "duration.txt").exists():
            video.parent.mkdir(parents=True, exist_ok=True)
            video.write_bytes(fetch(f"{MEDIA}/VideoFlash/{clip}.flv"))
            duration, _ = probe(video)
            result["n_frames"], result["sample_fps"] = extract_frames(
                video, frames_dir, args.fps, args.max_frames, args.max_side, duration)
            (frames_dir / "duration.txt").write_text(f"{duration} {result['sample_fps']}")
            video.unlink()
            result["duration"] = duration
        else:
            duration, fps = map(float, (frames_dir / "duration.txt").read_text().split())
            result.update(n_frames=len(existing), duration=duration, sample_fps=fps)
        if result["n_frames"] == 0 and result["status"] == "ok":
            result["status"] = "no_frames"
    except Exception as err:  # network or ffmpeg error: recorded, the clip is retried on the next run
        result["status"] = f"error: {getattr(err, 'stderr', None) or err}"[:200]
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--fps", type=float, default=2.0, help="target sampling rate (frames per second)")
    parser.add_argument("--max-frames", type=int, default=16, help="cap on frames per clip")
    parser.add_argument("--max-side", type=int, default=448, help="longest side of saved frames, in pixels")
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--limit", type=int, default=None, help="only process the first N clips")
    args = parser.parse_args()
    args.out_dir.mkdir(parents=True, exist_ok=True)

    clips = clip_table()
    if args.limit:
        clips = clips.groupby("mode").head(max(1, args.limit // 3)).reset_index(drop=True)

    rows = clips.to_dict("records")
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        results = list(pool.map(lambda r: process_clip(r, args.out_dir, args), rows))
    manifest = clips.merge(pd.DataFrame(results), on="uid", how="left")
    manifest.to_csv(args.out_dir / "manifest.csv", index=False)
    print(manifest["status"].str.slice(0, 40).value_counts().to_string())
    print(manifest.groupby("mode").agg(actors=("actor", "nunique"), clips=("uid", "size")).to_string())
    print(f"frames per clip: median {manifest['n_frames'].median():.0f}, max {manifest['n_frames'].max()} | "
          f"duration median {manifest['duration'].median():.2f} s | raters per condition (median): "
          f"{ {c: int(manifest[f'{c}_n'].median()) for c in CONDITIONS.values()} }")
    print(f"wrote {args.out_dir / 'manifest.csv'} ({len(manifest)} clips)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile preprocess_cremad_views.py
"""Extra visual views of CREMA-D for the input ablation of the visual probe (experiment A, folded into B).

The original frames (preprocess_cremad.py, view A0: 2 fps, longest side 448) stay as they are. This script adds:
    A1  frames_4fps/<actor>/<clip>/NNN.jpg   4 fps (twice the frames), longest side 448, same cap of 16 frames
    A2  faces/<actor>/<clip>/NNN.jpg         face crops at A0's timestamps (same sampling rate per clip), cut from
                                             the source frames (480x360) and resized to FACE_SIDE x FACE_SIDE
A3 (full frames + face crops) needs no new files: the run notebook passes A0's frames and A2's crops as two videos.

Face crops. OpenCV's frontal-face Haar cascade on every frame; the largest face per frame; one box per clip (median of
the per-frame boxes, so the crop does not jitter between frames), widened by MARGIN around its centre to keep chin and
forehead, made square, and shifted to stay inside the frame. Clips with no detected face in any frame get a central
square crop and status `face_fallback`. FACE_SIDE = 336, so a crop is 12 x 12 patches of 28 pixels after the model's
resizing, about the same number of visual tokens as a full frame of A0 (a 448x336 frame under the 448*252 pixel cap):
the face gets the token budget the whole frame had. The source video is 480x360, so the crop is upsampled; it adds
tokens per face, not new pixel detail.

Output (paths relative to <out>): the folders above, views.csv (one row per clip: paths, frame counts, rates, face
box, frames with a detected face, status) and face_samples.jpg (48 random crops, for a visual check).

Usage (needs ffmpeg/ffprobe on PATH and opencv-python; downloads the videos again):
    python analysis/preprocess_cremad_views.py --manifest /data/cremad_mm/manifest.csv --out-dir /data/cremad_views
"""

from __future__ import annotations

import argparse
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from preprocess_cremad import MEDIA, extract_frames, fetch, probe

FACE_SIDE = 336
MARGIN = 1.6
FPS_A1 = 4.0
MAX_FRAMES, MAX_SIDE = 16, 448  # as for A0
CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def face_box(frames: list[np.ndarray]) -> tuple[tuple[int, int, int], int]:
    """(centre x, centre y, side) of the square crop for a clip, and the number of frames with a detected face."""
    h, w = frames[0].shape[:2]
    found = []
    for f in frames:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        faces = CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(h // 6, h // 6))
        if len(faces):
            x, y, bw, bh = max(faces, key=lambda b: b[2] * b[3])
            found.append((x + bw / 2, y + bh / 2, max(bw, bh)))
    if found:
        cx, cy, side = np.median(np.array(found), 0)
    else:
        cx, cy, side = w / 2, h / 2, 0.6 * h / MARGIN
    side = int(round(min(MARGIN * side, h, w)))
    cx = int(round(min(max(cx, side / 2), w - side / 2)))
    cy = int(round(min(max(cy, side / 2), h - side / 2)))
    return (cx, cy, side), len(found)


def crop(frame: np.ndarray, box: tuple[int, int, int]) -> np.ndarray:
    cx, cy, side = box
    x0, y0 = cx - side // 2, cy - side // 2
    return cv2.resize(frame[y0:y0 + side, x0:x0 + side], (FACE_SIDE, FACE_SIDE), interpolation=cv2.INTER_CUBIC)


def faces_at(video: Path, dst: Path, fps: float) -> tuple[int, tuple[int, int, int], int]:
    """Source-resolution frames at `fps` (A0's rate for this clip), cropped to one face box per clip."""
    tmp = dst.with_name(dst.name + ".src")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(video), "-vf", f"fps={fps:.4f}",
                    "-q:v", "2", str(tmp / "%03d.png")], check=True)
    paths = sorted(tmp.glob("*.png"))[:MAX_FRAMES]
    if not paths:  # clip shorter than one sampling interval: its first frame, as in extract_frames
        subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(video), "-frames:v", "1",
                        str(tmp / "001.png")], check=True)
        paths = sorted(tmp.glob("*.png"))
    frames = [cv2.imread(str(p)) for p in paths]
    box, n_found = face_box(frames)
    out = dst.with_name(dst.name + ".tmp")
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    for k, f in enumerate(frames, 1):
        cv2.imwrite(str(out / f"{k:03d}.jpg"), crop(f, box), [cv2.IMWRITE_JPEG_QUALITY, 92])
    shutil.rmtree(dst, ignore_errors=True)
    out.rename(dst)
    shutil.rmtree(tmp)
    return len(frames), box, n_found


def process_clip(row: dict, out_dir: Path) -> dict:
    clip, actor = row["uid"], str(row["actor"])
    rel4, relf = Path("frames_4fps") / actor / clip, Path("faces") / actor / clip
    result = {"uid": clip, "frames4_dir": str(rel4), "faces_dir": str(relf), "status": "ok"}
    done = out_dir / relf / "box.txt"
    try:
        if done.exists():  # finished in an earlier run
            n4, fps4, nf, cx, cy, side, found = done.read_text().split()
        else:
            video = out_dir / "tmp_video" / f"{clip}.flv"
            video.parent.mkdir(parents=True, exist_ok=True)
            video.write_bytes(fetch(f"{MEDIA}/VideoFlash/{clip}.flv"))
            duration, _ = probe(video)
            n4, fps4 = extract_frames(video, out_dir / rel4, FPS_A1, MAX_FRAMES, MAX_SIDE, duration)
            nf, (cx, cy, side), found = faces_at(video, out_dir / relf, float(row["sample_fps"]))
            video.unlink()
            done.write_text(f"{n4} {fps4} {nf} {cx} {cy} {side} {found}")
        result.update(n_frames4=int(n4), sample_fps4=float(fps4), n_faces=int(nf), face_box=f"{cx} {cy} {side}",
                      face_frames_detected=int(found))
        if int(nf) != int(row["n_frames"]):
            result["status"] = f"frame_count_mismatch: {nf} vs {row['n_frames']}"
        elif int(found) == 0:
            result["status"] = "face_fallback"
    except Exception as err:  # network or ffmpeg error: recorded, the clip is retried on the next run
        result["status"] = f"error: {getattr(err, 'stderr', None) or err}"[:200]
    return result


def contact_sheet(out_dir: Path, views: pd.DataFrame, n: int = 48, seed: int = 0) -> None:
    ok = views[views.status.isin(["ok", "face_fallback"])]
    pick = ok.sample(min(n, len(ok)), random_state=seed)
    tiles = []
    for _, r in pick.iterrows():
        frames = sorted((out_dir / r.faces_dir).glob("*.jpg"))
        img = cv2.resize(cv2.imread(str(frames[len(frames) // 2])), (112, 112))
        if r.status == "face_fallback":
            cv2.rectangle(img, (0, 0), (111, 111), (0, 0, 255), 3)
        tiles.append(img)
    tiles += [np.zeros_like(tiles[0])] * (-len(tiles) % 8)
    rows = [np.hstack(tiles[i:i + 8]) for i in range(0, len(tiles), 8)]
    cv2.imwrite(str(out_dir / "face_samples.jpg"), np.vstack(rows))


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--manifest", type=Path, required=True, help="manifest.csv of preprocess_cremad.py (A0)")
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--limit", type=int, default=None, help="only process the first N clips")
    args = parser.parse_args()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    cv2.setNumThreads(1)

    manifest = pd.read_csv(args.manifest)
    clips = manifest[manifest.status == "ok"]
    if args.limit:
        clips = clips.head(args.limit)
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        results = list(pool.map(lambda r: process_clip(r, args.out_dir), clips.to_dict("records")))
    views = pd.DataFrame(results)
    views.to_csv(args.out_dir / "views.csv", index=False)
    shutil.rmtree(args.out_dir / "tmp_video", ignore_errors=True)
    print(views.status.str.slice(0, 40).value_counts().to_string())
    good = views[views.status.isin(["ok", "face_fallback"])]
    print(f"A1 frames per clip: median {good.n_frames4.median():.0f}, max {good.n_frames4.max()} | "
          f"face crops per clip: median {good.n_faces.median():.0f} | frames with a detected face: "
          f"{good.face_frames_detected.sum() / good.n_faces.sum():.1%} | face crop side (source px): median "
          f"{good.face_box.str.split().str[2].astype(int).median():.0f}")
    if len(good):
        contact_sheet(args.out_dir, good)
    print(f"wrote {args.out_dir / 'views.csv'} ({len(views)} clips)")


if __name__ == "__main__":
    main()

In [ ]:
import glob, os
manifest = [h for k in range(6) for h in glob.glob("/kaggle/input/" + "*/" * k + "cremad_mm/manifest.csv")][0]
# resume: copy an earlier run of this notebook (attached as input) into the working folder
for src in [h for k in range(6) for h in glob.glob("/kaggle/input/" + "*/" * k + "cremad_views")]:
    if not os.path.exists("/kaggle/working/cremad_views"):
        shutil.copytree(src, "/kaggle/working/cremad_views")
        print("copied earlier run from", src)
print("A0 manifest:", manifest)

In [ ]:
!python preprocess_cremad_views.py --manifest {manifest} --out-dir /kaggle/working/cremad_views --workers 16

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/cremad_views/face_samples.jpg"))
print(subprocess.run(["du", "-sh", "/kaggle/working/cremad_views"], capture_output=True, text=True).stdout)